In [1]:
print("hello world")


hello world


In [1]:
import os
from dotenv import load_dotenv

# LangChain imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.documents import Document

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate



C:\Users\User\AppData\Local\Temp\ipykernel_15520\2287087938.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\User\OneDrive\Desktop\FinalProjectRAG\Lanchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# =========================================================
# LOAD DOCUMENTS
# =========================================================

print("\nLoading documents...\n")

documents = []

data_folder = "data"

for file_name in os.listdir(data_folder):

    file_path = os.path.join(data_folder, file_name)

    loader = TextLoader(file_path, encoding="utf-8")

    docs = loader.load()

    documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")





Loading documents...



FileNotFoundError: [WinError 3] The system cannot find the path specified: 'data'

In [3]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\User\OneDrive\Desktop\FinalProjectRAG\Lanchain\rag\data
['brand_rules.txt', 'communication_rules.txt', 'company_assistant.ipynb', 'tone_style.txt']


In [5]:
print("\nLoading documents...\n")

documents = []

data_folder = "."

for file_name in os.listdir(data_folder):

    if file_name.endswith(".txt"):
        file_path = os.path.join(data_folder, file_name)

        loader = TextLoader(file_path, encoding="utf-8")
        docs = loader.load()

        documents.extend(docs)

print(f"Total documents loaded: {len(documents)}")


Loading documents...

Total documents loaded: 3


In [6]:
# =========================================================
# CHUNK DOCUMENTS
# =========================================================

print("\nChunking documents...\n")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")


Chunking documents...

Total chunks created: 3


In [7]:
# =========================================================
# CREATE EMBEDDINGS
# =========================================================

print("\nCreating embeddings model...\n")

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)


# =========================================================
# CREATE CHROMA VECTOR DATABASE
# =========================================================

print("\nCreating Chroma vector store...\n")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="company_guidelines",
    persist_directory="chroma_db"
)

print("Vector database created successfully!")




Creating embeddings model...


Creating Chroma vector store...

Vector database created successfully!


In [8]:
# =========================================================
# CREATE RETRIEVER
# =========================================================

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# =========================================================
# USER INPUT
# =========================================================

user_input = """
hey bro,
i used chatgpt and github for this project
"""


print("\nUser Input:\n")
print(user_input)


User Input:


hey bro,
i used chatgpt and github for this project



In [9]:
# =========================================================
# RETRIEVE RELEVANT DOCUMENTS
# =========================================================

print("\nRetrieving relevant chunks...\n")

retrieved_docs = retriever.invoke(user_input)

for i, doc in enumerate(retrieved_docs, start=1):

    print(f"\n--- Chunk {i} ---\n")

    print(doc.page_content)


# =========================================================
# COMBINE RETRIEVED CONTEXT
# =========================================================

context = "\n\n".join([doc.page_content for doc in retrieved_docs])


Retrieving relevant chunks...


--- Chunk 1 ---

Company Brand Capitalization Guidelines

Always write company and product names using official capitalization.

Correct:
- OpenAI
- ChatGPT
- GitHub
- LinkedIn
- YouTube

Incorrect:
- openai
- chatgpt
- github
- linkedin
- youtube

--- Chunk 2 ---

Customer Communication Guidelines

Always refer to users respectfully.

Preferred terms:
- customer
- client
- user

Avoid:
- bro
- dude
- guy
- kid

Use professional and inclusive language.

--- Chunk 3 ---

Writing Tone Guidelines

The company tone should be:
- professional
- friendly
- concise

Avoid:
- slang
- sarcasm
- aggressive language


In [10]:
# =========================================================
# CREATE PROMPT
# =========================================================

prompt = ChatPromptTemplate.from_template(
    """
You are a professional company writing assistant.

Use the provided company guidelines to fix the user text.

Guidelines:
{context}

User Text:
{input}

Return only the corrected professional version.
"""
)


In [11]:
# =========================================================
# CREATE LLM
# =========================================================

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4.1-mini",
    temperature=0
)


# =========================================================
# CREATE FINAL CHAIN
# =========================================================

chain = prompt | llm



In [12]:
# =========================================================
# GENERATE RESPONSE
# =========================================================

print("\nGenerating final response...\n")

response = chain.invoke({
    "context": context,
    "input": user_input
})


# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n========== FINAL OUTPUT ==========\n")

print(response.content)



Generating final response...


========== FINAL OUTPUT ==========

Hello,  
I used ChatGPT and GitHub for this project.


In [13]:
#Exercise
import os
import urllib.request

from dotenv import load_dotenv

# =========================================================
# LANGCHAIN IMPORTS
# =========================================================

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [21]:
# LOAD ENV VARIABLES
# =========================================================

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")


from langchain_community.document_loaders import PyPDFLoader

pdf_path = r"C:\Users\User\OneDrive\Desktop\FinalProjectRAG\Lanchain\rag_practice\rag_vs_finetuning_paper.pdf"

loader = PyPDFLoader(pdf_path)

pages = loader.load()

print(f"PDF loaded successfully!")
print(f"Total pages: {len(pages)}")

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


PDF loaded successfully!
Total pages: 77


In [22]:
# =========================================================
# STEP 2 — LOAD PDF
# =========================================================

print("\nLoading PDF...\n")

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(f"Total pages loaded: {len(documents)}")


# =========================================================
# OPTIONAL — PREVIEW DOCUMENT
# =========================================================

print("\nFirst page preview:\n")

print(documents[0].page_content[:1000])


# =========================================================
# STEP 3 — SPLIT INTO CHUNKS
# =========================================================

print("\nChunking documents...\n")

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")


# =========================================================
# OPTIONAL — PREVIEW CHUNK
# =========================================================

print("\nFirst chunk preview:\n")

print(chunks[0].page_content)




Loading PDF...



Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


Total pages loaded: 77

First page preview:

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev
Punit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich
Yinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra
Igor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi
Alan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang
Ross Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang
Angela Fan Melanie Kambadur Sharan Narang Au

In [23]:
# =========================================================
# STEP 4 — CREATE EMBEDDING MODEL
# =========================================================

print("\nCreating embedding model...\n")

embedding_model = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY,
    model="text-embedding-3-small"
)


# =========================================================
# STEP 5 — CREATE CHROMA VECTOR DATABASE
# =========================================================

print("\nCreating vector database...\n")

persist_directory = os.path.join(
    os.getcwd(),
    "chroma_db"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print("Vector database created successfully!")




Creating embedding model...


Creating vector database...

Vector database created successfully!


In [24]:
# =========================================================
# STEP 6 — CREATE RETRIEVER
# =========================================================

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


# =========================================================
# STEP 7 — CREATE PROMPT TEMPLATE
# =========================================================

message = """
You are a helpful AI research assistant.

Answer the user's question using ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("human", message)
])


In [25]:
# =========================================================
# STEP 8 — CREATE LLM
# =========================================================

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4.1-mini",
    temperature=0
)


# =========================================================
# STEP 9 — CREATE RAG CHAIN
# =========================================================

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
)


In [26]:
# =========================================================
# STEP 10 — ASK QUESTION
# =========================================================

question = "Tell me more about Llama 2 architecture."

print("\nUser Question:\n")
print(question)


# =========================================================
# STEP 11 — GENERATE RESPONSE
# =========================================================

print("\nGenerating response...\n")

response = rag_chain.invoke(question)


# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n========== FINAL ANSWER ==========\n")

print(response.content)


User Question:

Tell me more about Llama 2 architecture.

Generating response...


========== FINAL ANSWER ==========

The provided context does not include specific details about the architecture of Llama 2. It mentions observations made during the development of Llama 2 and Llama 2-Chat, such as the emergence of tool usage and temporal organization of knowledge, but does not describe the model's architecture. For more detailed information about Llama 2's architecture, additional sources would be needed.


In [19]:
import os

pdf_path = r"C:\Users\User\OneDrive\Desktop\FinalProjectRAG\Lanchain\rag_practice\rag_vs_finetuning_paper.pdf"

print(os.path.exists(pdf_path))
print(os.path.getsize(pdf_path))

True
13661300
